# Assignment 1: Reproducible Neutrino Workflow

**PHYS690: Computational Methods for Physics Research**  
**Unit 1: Research Computing Foundations**

**Student name(s):** _replace this text with your name(s)_  
**Private GitHub repository URL:** _replace this text with your assignment repository link_

## Purpose

This assignment is about the full shape of a small computational research workflow on your local machine using VS Code. You will inspect a starter data set, organize a project repository, use shell and Git operations, load and transform CSV data with Python, create a reproducible result plot, and document how someone else can rerun your work.

The final product is your private GitHub repository in the `WM-PHYS690-Fall2026` organization, not just a completed notebook.

## Scientific setting

Neutrinos are light, electrically neutral particles that interact through the weak force. There are three flavors of neutrino, usually written as $\nu_e$, $\nu_\mu$, and $\nu_\tau$, associated with the electron, muon, and tau. In a long-baseline neutrino experiment, a beam of mostly muon neutrinos can be produced at one location and detected hundreds of kilometers away. If neutrinos oscillate between flavors, the observed energy spectrum can differ from a no-oscillation simulation.

This is the same broad physics program pursued by modern long-baseline experiments such as NOvA. In 2019, the NOvA Collaboration reported measurements of neutrino oscillation parameters using both neutrino and antineutrino data in [arXiv:1906.04907](https://arxiv.org/pdf/1906.04907). The High Energy Experimental group at William & Mary contributed to this experimental program. The data set in this assignment is greatly simplified, but the core idea is similar: compare an observed energy spectrum with a no-oscillation expectation and look for evidence of oscillation-like shape changes that are governed by fundamental oscillation parameters.

## Data files

The starter data for this assignment are the CSV files included in this assignment repository:

- `data/raw/neutrino_data.csv`: observed event energies;
- `data/raw/neutrino_simulation.csv`: simulated event energies from a no-oscillation Monte Carlo sample.

Each row contains one reconstructed neutrino energy in GeV. The simulation sample corresponds to more exposure than the observed data sample, so we will use an exposure scale factor before comparing event counts.

## Deliverable

Submit your private GitHub repository containing a reproducible analysis that runs locally in VS Code. Your repository should include:

- this completed notebook;
- a Python script at `scripts/make_neutrino_figure.py` that can reproduce the final figure from the command line;
- the input CSV files in a documented data directory;
- a final result figure saved from code;
- a `requirements.txt` file listing the Python packages needed to rerun the notebook;
- a `README.md` explaining how to rerun the work;
- a Git history showing meaningful progress through the assignment.

A reasonable repository layout is:

```text
assignment1-neutrino-workflow/
  .gitignore
  README.md
  requirements.txt
  data/
    raw/
      neutrino_data.csv
      neutrino_simulation.csv
  figures/
    neutrino_data_simulation_ratio.png
  notebooks/
    assignment1_reproducible_neutrino_workflow.ipynb
  scripts/
    make_neutrino_figure.py
```

## Part 1: Set up a local Python environment

Open your assignment repository folder in VS Code. If you need help installing or opening VS Code, start with this repository's [README](../README.md) and the course VS Code resource. Use `Terminal > New Terminal` to create and activate a local Python environment, then install the packages used here. Windows users should choose Git Bash as the VS Code terminal profile for course commands.

On macOS:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

On Windows Git Bash:

```bash
python -m venv .venv
source .venv/Scripts/activate
python -m pip install -r requirements.txt
```

After that, open this notebook in VS Code and select the `.venv` Python kernel. Then run the next cell.

In [ ]:
%matplotlib inline

from pathlib import Path
import subprocess
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## Part 2: Use the VS Code terminal to inspect and organize the project

Use the VS Code terminal for command-line work. These are the same kinds of commands used in research projects to move around, inspect files, create directories, and run Git.

Useful Bash-style commands on macOS Terminal or Windows Git Bash:

```bash
pwd
ls
mkdir -p data/raw figures notebooks scripts
```

The next cell gives a cross-platform check of the same basic information from inside Python.

In [ ]:
# Print the current working directory from Python.
print(Path.cwd())

# List files and directories in the current working directory.
for path in sorted(Path.cwd().iterdir()):
    marker = "/" if path.is_dir() else ""
    print(f"{path.name}{marker}")

The template repository already includes the starter CSV files in `data/raw/`. The next cell confirms the expected project directories and creates any missing folders.

In [ ]:
# Define standard project directories.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURE_DIR = PROJECT_ROOT / "figures"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPT_DIR = PROJECT_ROOT / "scripts"

# Create directories if they do not already exist.
for directory in [DATA_DIR, FIGURE_DIR, NOTEBOOK_DIR, SCRIPT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Keep local environments and notebook checkpoints out of Git.
gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_entries = [".venv/", ".ipynb_checkpoints/", "__pycache__/", ".DS_Store"]
existing_gitignore = gitignore_path.read_text() if gitignore_path.exists() else ""

with gitignore_path.open("a") as gitignore_file:
    if existing_gitignore and not existing_gitignore.endswith("\n"):
        gitignore_file.write("\n")
    for entry in gitignore_entries:
        if entry not in existing_gitignore.splitlines():
            gitignore_file.write(f"{entry}\n")

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Figure directory:", FIGURE_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Script directory:", SCRIPT_DIR)

In [ ]:
# Confirm that the expected directories and CSV files are present.
for path in sorted(PROJECT_ROOT.rglob("*")):
    if path.is_file() and len(path.relative_to(PROJECT_ROOT).parts) <= 3:
        print(path.relative_to(PROJECT_ROOT))

### Git checkpoint 1

At this stage, make sure you are working on a `submission` branch and make an early commit. If you cloned the private assignment repository from GitHub, you do **not** need to run `git init`; the repository is already a Git repository. Run these commands in the VS Code terminal from the root of your assignment repository.

```bash
git status
git switch -c submission
git add notebooks/assignment1_reproducible_neutrino_workflow.ipynb README.md
git commit -m "Start assignment 1 workflow"
git status
```

If the `submission` branch already exists, use `git switch submission` instead.

In [ ]:
# Use this cell to inspect the Git status of your working tree from Python.
_ = subprocess.run(["git", "status", "--short"], cwd=PROJECT_ROOT, check=False)

## Part 3: Load the CSV data

`neutrino_data.csv` contains reconstructed energies for observed muon-neutrino candidate events. `neutrino_simulation.csv` contains reconstructed energies from a no-oscillation Monte Carlo (MC) simulation of the same experiment. The simulation models the neutrino beam, neutrino interactions, detector geometry, and detector response, but it represents a larger exposure than the observed data sample. Later we will scale the simulated event counts before comparing them with the data.

The energies are in units of GeV = 1 billion (Giga) electron-volts (eV). In those units the proton has a mass of $0.938~\mathrm{GeV}/c^2$ $(E=mc^2)$. In particle physics we often choose a unit system in which $c=1$, which allows us to refer to masses, momenta, and energies using related units.

A histogram of event energies is an approximation to the underlying energy distribution. Each event has one reconstructed energy, and the histogram counts how many events fall into each energy range, or _bin_. Comparing the observed histogram with the scaled no-oscillation simulation will let us look for shape differences.

The CSV files do not have column headers, so we provide the column name `energy_GeV` when we read them. The result should be two `pandas` DataFrames.

In [ ]:
# Point to the input files in the organized data directory.
DATA_PATH = DATA_DIR / "neutrino_data.csv"
SIMULATION_PATH = DATA_DIR / "neutrino_simulation.csv"

# Stop with a useful message if the data files are not in the expected location.
missing_paths = [path for path in [DATA_PATH, SIMULATION_PATH] if not path.exists()]
if missing_paths:
    missing_names = ", ".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Missing input CSV file(s): {missing_names}")

# Read one energy value per row from each CSV file.
data_events = pd.read_csv(DATA_PATH, header=None, names=["energy_GeV"])
simulation_events = pd.read_csv(SIMULATION_PATH, header=None, names=["energy_GeV"])

# Add labels before combining the two samples.
data_events["sample"] = "data"
simulation_events["sample"] = "simulation"
events = pd.concat([data_events, simulation_events], ignore_index=True)

events.head()

In [ ]:
# Count rows and summarize the energy values for each sample.
summary = events.groupby("sample")["energy_GeV"].agg(["count", "min", "max", "mean", "std"])
summary

### Interpretation prompt

In a markdown cell below this one, briefly answer:

- How many observed events are in the data sample?
- How many simulated events are in the simulation sample?
- What is one thing you notice from the minimum, maximum, or mean values?

_Your answer here._

## Part 4: Transform the data for analysis

We will use an energy range from 0 to 20 GeV.  Values outside this range are not deleted from the input files; instead, we create a transformed analysis table that records which events are used for the comparison.

In [ ]:
# Define the analysis window and mark whether each row is inside it.
ENERGY_MIN_GEV = 0.0
ENERGY_MAX_GEV = 20.0

events["in_analysis_window"] = events["energy_GeV"].between(ENERGY_MIN_GEV, ENERGY_MAX_GEV)

# Count how many rows are inside and outside the analysis window.
window_counts = (
    events.groupby(["sample", "in_analysis_window"])
    .size()
    .rename("n_events")
    .reset_index()
)

window_counts

In [ ]:
# Keep only events in the analysis window for the histogram comparison.
analysis_events = events.loc[events["in_analysis_window"]].copy()

# Split the combined table back into observed and simulated energy arrays.
data_energy = analysis_events.loc[analysis_events["sample"] == "data", "energy_GeV"].to_numpy()
simulation_energy = analysis_events.loc[analysis_events["sample"] == "simulation", "energy_GeV"].to_numpy()

print(f"Data events in analysis window: {len(data_energy)}")
print(f"Simulation events in analysis window: {len(simulation_energy)}")

### Interpretation prompt

In a markdown cell below this one, explain why it is better to document the analysis window than to silently ignore events outside the plotting range.

_Your answer here._

## Part 5: Histogram the observed and simulated samples

The simulation sample has more exposure than the observed data sample. Use the exposure scale factor below to compare simulated counts to observed counts on the same scale. This cell is provided so that everyone uses the same data/simulation histogram for the rest of the assignment.

In [ ]:
# Choose the histogram binning.
N_BINS = 20
bin_edges = np.linspace(ENERGY_MIN_GEV, ENERGY_MAX_GEV, N_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_widths = np.diff(bin_edges)

# Histogram the observed and simulated energies with identical bin edges.
data_counts, _ = np.histogram(data_energy, bins=bin_edges)
simulation_counts, _ = np.histogram(simulation_energy, bins=bin_edges)

# Scale the simulation to the data exposure before comparing counts.
EXPOSURE_SCALE = 0.1

scaled_simulation_counts = EXPOSURE_SCALE * simulation_counts

histogram_table = pd.DataFrame(
    {
        "energy_center_GeV": bin_centers,
        "data_counts": data_counts,
        "scaled_simulation_counts": scaled_simulation_counts,
    }
)

histogram_table.head()

### Try this before moving on

Look at the table above and make sure the binning makes sense. For this assignment, leave `N_BINS = 20` in your submitted notebook and script so that everyone uses the same data/simulation ratio. In your README, briefly explain that the comparison uses 20 bins from 0 to 20 GeV.

## Part 6: Build the data/simulation ratio

The ratio highlights shape differences between the observed data and the no-oscillation simulation. Since each histogram bin is a counting measurement, estimate the statistical uncertainty using counting uncertainties. Bins with zero counts should be treated carefully. This cell is also provided so that everyone uses the same ratio and uncertainty definitions.

For a ratio $R = D / S$, where $D$ is the data count and $S$ is the scaled simulation count, we use the approximate relative uncertainty

$$\left(\frac{\sigma_R}{R}\right)^2 \approx \frac{1}{D} + \frac{1}{N_\mathrm{sim}},$$

where $N_\mathrm{sim}$ is the unscaled simulation count in the same bin.

In [ ]:
# Compute data/simulation ratios only where the denominator is nonzero.
ratio = np.divide(
    data_counts,
    scaled_simulation_counts,
    out=np.full_like(scaled_simulation_counts, np.nan, dtype=float),
    where=scaled_simulation_counts > 0,
)

# Propagate approximate Poisson counting uncertainty for the ratio.
data_relative_variance = np.divide(
    1.0,
    data_counts,
    out=np.full_like(data_counts, np.nan, dtype=float),
    where=data_counts > 0,
)
simulation_relative_variance = np.divide(
    1.0,
    simulation_counts,
    out=np.full_like(simulation_counts, np.nan, dtype=float),
    where=simulation_counts > 0,
)
ratio_uncertainty = ratio * np.sqrt(data_relative_variance + simulation_relative_variance)

# Store the transformed result in a table for inspection.
ratio_table = histogram_table.copy()
ratio_table["data_simulation_ratio"] = ratio
ratio_table["ratio_uncertainty"] = ratio_uncertainty

ratio_table

Plot the data/simulation ratio from `ratio_table` to inspect the shape before comparing it with an oscillation model.

In [ ]:
# Plot the data/simulation ratio using the values stored in ratio_table.
fig, ax = plt.subplots(figsize=(7, 4))

ax.axhline(1.0, color="0.5", lw=1, ls=":")
ax.errorbar(
    ratio_table["energy_center_GeV"],
    ratio_table["data_simulation_ratio"],
    yerr=ratio_table["ratio_uncertainty"],
    fmt="o",
    capsize=2,
)
ax.set_xlabel("neutrino energy (GeV)")
ax.set_ylabel("data / simulation")
ax.set_ylim(0.0, 1.4)
ax.set_title("Data/simulation ratio")

fig.tight_layout()
plt.show()

## Part 7: Compare with a simple oscillation curve

The ratio you computed is useful because neutrino oscillations change the shape of the observed energy spectrum. For this assignment, we will use a simplified two-neutrino picture with only $\nu_\mu$ and $\nu_\tau$. This is not the full three-flavor theory, but it captures the main idea needed for the plot.

### Mass and flavor eigenstates

Neutrinos are produced and detected through the weak interaction as **flavor eigenstates**. A muon neutrino, $\nu_\mu$, is the flavor state associated with a muon, while a tau neutrino, $\nu_\tau$, is associated with a tau. However, if neutrinos have mass, the states with definite energy in free propagation are **mass eigenstates**, which we call $\nu_1$ and $\nu_2$.

The mass eigenstates are eigenstates of the Hamiltonian, the quantum-mechanical energy operator:

$$\mathcal{H}|\nu_1\rangle = E_1 |\nu_1\rangle, \qquad \mathcal{H}|\nu_2\rangle = E_2 |\nu_2\rangle.$$

The important point is that flavor eigenstates do not have to be the same as mass eigenstates. In the two-neutrino approximation, the two bases can be related by one mixing angle $\theta$:

$$
\begin{bmatrix}
\nu_\mu \\
\nu_\tau
\end{bmatrix}
=
\begin{bmatrix}
\cos\theta & \sin\theta \\
-\sin\theta & \cos\theta
\end{bmatrix}
\begin{bmatrix}
\nu_1 \\
\nu_2
\end{bmatrix}.
$$

This means a neutrino can be born as a definite flavor, but it propagates as a superposition of states with different masses and therefore different phases.

### Born as flavor, propagated as mass

A beam made from pion decays starts with many muon neutrinos. At the source, a muon neutrino can be written as

$$|\nu_\mu(0)\rangle = \cos\theta |\nu_1\rangle + \sin\theta |\nu_2\rangle.$$

After time $t$, each mass component accumulates its own quantum phase:

$$|\nu_\mu(t)\rangle = e^{-iE_1t}\cos\theta |\nu_1\rangle + e^{-iE_2t}\sin\theta |\nu_2\rangle.$$

To ask whether the neutrino is still a $\nu_\mu$ at the detector, we compute the overlap with the original $\nu_\mu$ flavor state and square its magnitude:

$$P(\nu_\mu \to \nu_\mu) = |\langle \nu_\mu(0)|\nu_\mu(t)\rangle|^2 = \left|e^{-iE_1t}\cos^2\theta + e^{-iE_2t}\sin^2\theta\right|^2.$$

The two phase factors can interfere. After rewriting the phase difference in terms of the distance traveled $L$, neutrino energy $E$, and squared-mass difference $\Delta m^2 \equiv m_2^2 - m_1^2$, the survival probability becomes

$$P(\nu_\mu \to \nu_\mu) = 1 - \sin^2 2\theta \, \sin^2\left(1.27 \frac{\Delta m^2 L}{E}\right).$$

Here $\Delta m^2$ is in $\mathrm{eV}^2$, $L$ is in km, and $E$ is in GeV. The numerical factor 1.27 comes from the unit conversion. The parameter $\sin^2 2\theta$ controls how deep the disappearance dip can be, and $\Delta m^2$ controls where the oscillation pattern appears in energy. Do not do a formal fit in this assignment. The goal is to make a clean, reproducible comparison plot and describe the data qualitatively.

In the next code cell, translate this equation into a Python function that can accept a NumPy array of energies and return a NumPy array of survival probabilities.

In [ ]:
# TODO: Define a function that returns the survival probability.
# Inputs should include energy_GeV, delta_m2, sin2_2theta, and baseline_km.
# Use the formula in the markdown cell above.
def survival_probability(energy_GeV, delta_m2, sin2_2theta, baseline_km):
    ...


# TODO: Choose starting parameter values for the comparison curve.
# Use values near Delta m^2 = 2.5e-3 eV^2 and sin^2(2 theta) = 0.95.
DELTA_M2 = ...
SIN2_2THETA = ...
BASELINE_KM = 735.0

# TODO: Create an energy grid from about 0.2 GeV to ENERGY_MAX_GEV.
energy_grid = ...

# TODO: Evaluate your survival_probability function on the energy grid.
survival_curve = ...

## Part 8: Adjust the model qualitatively and save the result figure

Your final plot should be reproducible from the CSV files and the code in this notebook. It should overlay the data/simulation ratio with uncertainties and your qualitative oscillation-model curve. It should have labeled axes, a legend, and enough information in the caption or README for a reader to understand what was compared.

Before saving the final figure, explore how the oscillation model changes when you modify its parameters. Do **not** perform a formal fit or calculate a best-fit value. Instead, vary the parameters by hand and choose values that qualitatively describe the main shape of the data/simulation ratio.

In the survival-probability model, $\Delta m^2$ changes where the dip appears as a function of energy, while $\sin^2 2\theta$ changes the depth of the dip. The baseline is fixed at $L = 735~\mathrm{km}$ for this assignment.

First, plot the model for several parameter choices. Use values such as

$$\Delta m^2 = 5 \times \{10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}\}~\mathrm{eV}^2$$

and

$$\sin^2 2\theta = \{0.5, 0.75, 1.0\}.$$

Make a 2x2 grid with one $\Delta m^2$ value per panel and the three $\sin^2 2\theta$ values shown as separate curves. Use this scan to build intuition about which parameter controls the position and depth of the oscillation feature.

In [ ]:
# TODO: Explore the survival-probability model for several parameter choices.
# Use the Delta m^2 and sin^2(2 theta) values listed above.
delta_m2_values = ...
sin2_2theta_values = ...

# TODO: Make a 2x2 grid of axes with plt.subplots.
fig_scan, axes = ...

# TODO: For each Delta m^2 value, plot three curves with different
# sin^2(2 theta) values. Add panel titles, axis labels, and a legend.
...

plt.show()

Now choose your own qualitative parameter values for the final comparison. Start near the defaults, adjust one parameter at a time, and rerun the final plotting cell until the curve gives a reasonable visual description of the ratio points. Your goal is a defensible by-eye comparison, not a numerical optimum.

In [ ]:
# TODO: Replace these with your by-eye qualitative choices.
DELTA_M2 = ...
SIN2_2THETA = ...

# TODO: Recompute survival_curve using your chosen parameters.
survival_curve = ...

print(f"Qualitative model choice: Delta m^2 = {DELTA_M2:.3g} eV^2")
print(f"Qualitative model choice: sin^2(2 theta) = {SIN2_2THETA:.3g}")

Your final figure should include:

- data/simulation ratio points from `ratio_table`;
- vertical uncertainty bars from `ratio_table["ratio_uncertainty"]`;
- a horizontal reference line at ratio = 1;
- your qualitative model curve from `survival_curve`;
- labeled axes, a legend, and a descriptive title;
- output saved to `figures/neutrino_data_simulation_ratio.png`.

In [ ]:
# TODO: Create a single-panel final result figure.
# Overlay the data/simulation ratio with uncertainty bars and your qualitative model curve.
# Include labeled axes, legends, and a descriptive title.
fig, ax = ...

# TODO: Draw the ratio points from ratio_table and your survival-probability curve on ax.
...

# Save the figure into the repository so it can be committed.
FIGURE_PATH = FIGURE_DIR / "neutrino_data_simulation_ratio.png"
# TODO: Save the figure to FIGURE_PATH with good resolution.
...
plt.show()

print(f"Saved figure to: {FIGURE_PATH}")

In [ ]:
# Confirm that the result figure exists and has a nonzero file size.
for path in sorted(FIGURE_DIR.iterdir()):
    size_kb = path.stat().st_size / 1024
    print(f"{path.name}: {size_kb:.1f} KB")

### Interpretation prompt

In a markdown cell below this one, briefly describe your final plot. What does the ratio show that is less obvious from the count histogram alone? What values of $\Delta m^2$ and $\sin^2 2\theta$ did you choose by eye, and why do they qualitatively describe the data better than the starting values?

_Your answer here._

## Part 9: Write a command-line reproduction script

A reproducible analysis should not depend only on notebook state. Write and submit a Python script named `scripts/make_neutrino_figure.py` that reproduces the same final figure generated in Part 8, including your by-eye qualitative choices for $\Delta m^2$ and $\sin^2 2\theta$. The script should read the CSV files from `data/raw/`, repeat the histogram, ratio, uncertainty, and oscillation-curve calculations, and save `figures/neutrino_data_simulation_ratio.png` when executed from the command line.

Your script should be runnable from the root of the repository with:

```bash
python scripts/make_neutrino_figure.py
```

It should not require the notebook to be open or previously run. Use functions where they make the script clearer, include a small `main()` function, and protect command-line execution with `if __name__ == "__main__":`. After running the script, confirm that the figure file was created or updated. Commit this script to the GitHub repository and describe how to execute it in your README.

After you write the script, run the command below from the VS Code terminal. Then rerun the following notebook cell to confirm that the figure exists and has nonzero size.

```bash
python scripts/make_neutrino_figure.py
```

In [ ]:
# Verify that the command-line script produced the expected figure file.
script_figure_path = PROJECT_ROOT / "figures" / "neutrino_data_simulation_ratio.png"
if not script_figure_path.exists():
    raise FileNotFoundError(f"Expected figure not found: {script_figure_path}")

size_kb = script_figure_path.stat().st_size / 1024
if size_kb == 0:
    raise ValueError(f"Figure file is empty: {script_figure_path}")

print(f"Verified figure: {script_figure_path} ({size_kb:.1f} KB)")

## Part 10: Document reproducibility

A reproducible repository needs instructions. Your `README.md` should tell a reader what the project does, where the data came from, what software was used, how to open the project in VS Code, how to select the Python environment, how to rerun the notebook, how to run the command-line script, and where the final plot is saved.

In [ ]:
# Record a small software manifest for your README.
software_manifest = pd.DataFrame(
    {
        "tool": ["Python", "NumPy", "pandas", "Matplotlib"],
        "version": [sys.version.split()[0], np.__version__, pd.__version__, matplotlib.__version__],
    }
)

software_manifest

Your `README.md` should include, in your own words:

- project title and short scientific goal;
- repository layout;
- input data files and where they are stored;
- Python packages needed to rerun the analysis;
- how `requirements.txt` was created;
- step-by-step instructions to open the repository in VS Code, select the Python kernel, and rerun all notebook cells;
- the command needed to run `scripts/make_neutrino_figure.py` from the terminal;
- name and path of the generated figure;
- one-paragraph interpretation of the result;
- any substantial use of generative AI, if applicable.

## Part 11: Use Git and submit through GitHub

Make commits as you work. Your final repository should show at least three meaningful commits, for example:

1. initial project structure and data files;
2. completed notebook analysis and saved figure;
3. command-line reproduction script;
4. final README and cleanup.

Suggested final commands:

```bash
python -m pip freeze > requirements.txt
git status
git add README.md requirements.txt data/raw/*.csv figures/neutrino_data_simulation_ratio.png notebooks/assignment1_reproducible_neutrino_workflow.ipynb scripts/make_neutrino_figure.py
git commit -m "Complete reproducible neutrino workflow"
git status
git log --oneline --max-count=5
git push -u origin submission
```

Finally, submit a "pull request" from the `submission` branch into `main` in your private assignment repository.  This will inform Prof. Stevens that your submission is ready to be evaluated.

## Final checklist

Before submitting your GitHub repository URL, check that:

- the notebook runs from top to bottom after restarting the VS Code notebook kernel;
- `python scripts/make_neutrino_figure.py` runs from the terminal and reproduces the final figure;
- the CSV files are read from `data/raw/`, not typed manually into the notebook;
- the final figure is generated by code and saved in `figures/`;
- the README explains how to rerun the analysis;
- `.venv/` is listed in `.gitignore` and is not committed;
- `requirements.txt` is committed;
- the repository contains meaningful Git commits;
- the GitHub repository is private and located in `WM-PHYS690-Fall2026`;
- the instructor has access to the repository or submitted pull request.